In [27]:
# Hyperparameters
larger_grid_weight_choice = 1.2
shrink_to_center_prob_choice = 0.05
less_colors_weight_choice = 0.6
num_samples_choice = 10**9
batch_size_choice = 32
num_epochs_choice = 20
learning_rate_choice = 1e-4
dropout_prob_choice = 0.5

In [28]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import random
from torchvision import transforms
from PIL import Image
import timm 
import ast
from tqdm import tqdm
from sklearn.preprocessing import OneHotEncoder
import importlib.util

DSL_PATH = 'dsl.py'  # Update this path as needed
CONSTANTS_PATH = 'constants.py'  # Update this path as needed

# Load the dsl module dynamically
spec = importlib.util.spec_from_file_location("dsl_module", DSL_PATH)
dsl_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dsl_module)

# Load the constants module dynamically
spec_const = importlib.util.spec_from_file_location("constants_module", CONSTANTS_PATH)
constants_module = importlib.util.module_from_spec(spec_const)
spec_const.loader.exec_module(constants_module)

def parse_dsl_functions(dsl_path):
    """Parses the dsl.py file to extract functions with their argument and return types."""
    with open(dsl_path, 'r') as file:
        node = ast.parse(file.read(), filename=dsl_path)

    functions = []
    for n in node.body:
        if isinstance(n, ast.FunctionDef):
            func_name = n.name
            args = []
            # Extract argument names and types
            for arg in n.args.args:
                arg_name = arg.arg
                arg_type = None
                if arg.annotation:
                    arg_type = ast.unparse(arg.annotation)
                args.append((arg_name, arg_type))

            # Extract return type
            return_type = None
            if n.returns:
                return_type = ast.unparse(n.returns)

            # Get the function object from the module
            func = getattr(dsl_module, func_name, None)
            if func is not None:
                functions.append({
                    'name': func_name,
                    'function': func,
                    'args': args,
                    'return_type': return_type
                })
    return functions

dsl_functions = parse_dsl_functions(DSL_PATH)

In [29]:
constants_dict = {name: getattr(constants_module, name) for name in dir(constants_module) if not name.startswith('__')}

type_to_constants = {
    'Boolean': [True, False],
    'Integer': [v for v in constants_dict.values() if isinstance(v, int)],
    'IntegerTuple': [v for v in constants_dict.values() if isinstance(v, tuple) and all(isinstance(i, int) for i in v)],
}

In [30]:
def generate_sample(
    larger_grid_weight=larger_grid_weight_choice,
    shrink_to_center_prob=shrink_to_center_prob_choice,
    less_colors_weight=less_colors_weight_choice
):
    # Function to convert a numpy array to a tuple of tuples
    def to_tuple_of_tuples(grid):
        return tuple(tuple(row) for row in grid)

    # Generate numbers for grid dimensions
    numbers = np.arange(2, 31)
    weights = larger_grid_weight ** numbers
    probabilities = weights / weights.sum()

    PRE_width = np.random.choice(numbers, p=probabilities)
    PRE_length = np.random.choice(numbers, p=probabilities)

    # Choose new_number_of_colors based on less_colors_weight
    possible_colors = np.arange(1, 11)
    color_weights = less_colors_weight ** (10 - possible_colors)
    color_probabilities = color_weights / color_weights.sum()

    new_number_of_colors = np.random.choice(possible_colors, p=color_probabilities)

    # Generate PRE grid using integers from 0 to new_number_of_colors
    PRE = np.random.randint(0, new_number_of_colors + 1, size=(PRE_width, PRE_length), dtype=int)

    # Shrink the grid to a central rectangle with some probability
    if np.random.rand() < shrink_to_center_prob:
        new_width = np.random.randint(1, PRE_width + 1)
        new_length = np.random.randint(1, PRE_length + 1)
        row_start = (PRE_width - new_width) // 2
        col_start = (PRE_length - new_length) // 2
        
        # Create a new grid with zeros and place the smaller grid in the center
        new_PRE = np.zeros((PRE_width, PRE_length), dtype=int)
        new_PRE[row_start:row_start+new_width, col_start:col_start+new_length] = \
            np.random.randint(0, new_number_of_colors + 1, size=(new_width, new_length))
        PRE = new_PRE

    # Count occurrences of each color (from 1 to new_number_of_colors)
    color_counts = {color: np.sum(PRE == color) for color in range(1, new_number_of_colors + 1)}

    # Sort colors based on frequency (most frequent first)
    sorted_colors = sorted(color_counts, key=color_counts.get, reverse=True)

    # Create a mapping from the original colors to new ones
    color_mapping = {old_color: new_color+1 for new_color, old_color in enumerate(sorted_colors)}

    # Map the colors in PRE according to the new color mapping
    for old_color, new_color in color_mapping.items():
        PRE[PRE == old_color] = -new_color  # Temporarily use negative values to avoid conflicts
    PRE = np.abs(PRE)  # Restore the new colors

    # Convert PRE from numpy array to a tuple of tuples
    PRE = to_tuple_of_tuples(PRE)
    # Store PRE in variables and initialize grid as a copy of PRE
    
    variables = {'PRE': PRE}
    variable_types = {'PRE': 'Grid'}
    grid = PRE
    n_values = np.arange(1, 51)
    probabilities = n_values / n_values.sum()  # Probability proportional to n
    minimum_num_transformations = np.random.choice(n_values, p=probabilities)
    transformations_done = 0
    while True:
        func_info = random.choice(dsl_functions)
        func_name = func_info['name']
        func = func_info['function']
        arg_info = func_info['args']
        return_type = func_info['return_type']

        # Get arguments for the function
        args = []
        for arg_name, arg_type in arg_info:
            arg_value = None
            matching_vars = [variables[var_name] for var_name, var_type in variable_types.items() if var_type == arg_type]
            possible_values = type_to_constants.get(arg_type, [])
            combined_values = matching_vars + possible_values
            if combined_values:
                arg_value = random.choice(combined_values)
            if arg_value is None:
                continue
            args.append(arg_value)
        try:
            output = func(*args)
            # After minimum number of transformations, check if we can end
            if transformations_done >= minimum_num_transformations:
                if return_type == 'Grid':
                    grid = output
                    transformations_done += 1
                    break
                else:
                    variables[f'var_{transformations_done}'] = output
                    variable_types[f'var_{transformations_done}'] = return_type
                    transformations_done += 1
            else:
                variables[f'var_{transformations_done}'] = output
                variable_types[f'var_{transformations_done}'] = return_type
                transformations_done += 1
        except Exception as e:
            continue

        # Check if we have reached the desired number of transformations and can end
        if transformations_done >= minimum_num_transformations and return_type == 'Grid':
            break
    
    POST = grid

    # Ensure 2d. [I'm not sure why I need this].
    def get_num_dimensions(t):
        if isinstance(t, tuple) and len(t) > 0:
            return 1 + get_num_dimensions(t[0])
        return 0
    
    if(get_num_dimensions(POST) < 2):
        return generate_sample()
    
    # Ensure homogenous shape. [I'm not sure why I need this].
    def ensure_homogeneous_shape(grid):
        max_length = max(len(row) for row in grid)
        return tuple(tuple(row + (11,) * (max_length - len(row))) for row in grid)
    POST = ensure_homogeneous_shape(POST)

    # Ensure POST is a valid grid by checking dimensions and applying pooling if necessary
    if len(POST) > 30 or len(POST[0]) > 30:
        # Apply pooling to reduce the dimensions to <= 30 by taking the upper left corner of each subrectangle
        def upper_left_pooling(grid, max_dim=30):
            grid_np = np.array(grid)
            if grid_np.shape[0] > max_dim:
                split_size = int(np.ceil(grid_np.shape[0] / max_dim))
                grid_np = grid_np[::split_size, :]
            if grid_np.shape[1] > max_dim:
                split_size = int(np.ceil(grid_np.shape[1] / max_dim))
                grid_np = grid_np[:, ::split_size]
            return to_tuple_of_tuples(grid_np)
        POST = upper_left_pooling(POST)
        transformations_done += 1

    # Pad PRE and POST to 30x30 with 11s (convert them to numpy for padding)
    PRE_np = np.array(PRE, dtype=int)
    POST_np = np.array(POST, dtype=int)

    # Check for negative values in POST and rerun if true.
    if(np.any(POST_np < 0) or np.any(POST_np > new_number_of_colors)):
        return generate_sample()

    PRE_np = np.pad(PRE_np, ((0, 30 - PRE_np.shape[0]), (0, 30 - PRE_np.shape[1])), mode='constant', constant_values=11)
    POST_np = np.pad(POST_np, ((0, 30 - POST_np.shape[0]), (0, 30 - POST_np.shape[1])), mode='constant', constant_values=11)

    # Concatenate PRE and POST to form a 30x60 grid
    sample_grid_np = np.hstack((PRE_np, POST_np))
    sample_grid = to_tuple_of_tuples(sample_grid_np)

    # Label is the number of transformations applied
    label = transformations_done
        
    return sample_grid, label


In [31]:
# One-hot encoding for preprocessing; 12 channel CNN input
def one_hot_encode(sample):
    sample = np.array(sample)
    encoder = OneHotEncoder(categories=[range(12)], sparse_output=False, dtype=np.float32)
    encoded = encoder.fit_transform(sample.reshape(-1, 1)).reshape(sample.shape[0], sample.shape[1], 12)
    return encoded

# Dataset class
class ARCDataset(Dataset):
    def __init__(self, num_samples):
        self.samples = []
        self.labels = []

        for _ in tqdm(range(num_samples), desc="Generating samples"):
            sample, label = generate_sample()  # Assuming this function generates 30x60 grids and labels
            sample = one_hot_encode(sample)    # Preprocess by one-hot encoding
            self.samples.append(sample)
            self.labels.append(label)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        label = self.labels[idx]
        
        # Convert sample and label to tensors
        sample = torch.tensor(sample, dtype=torch.float32).permute(2, 0, 1)  # Shape: (12, 30, 60)
        label = torch.tensor(label, dtype=torch.float32)
        return sample, label

# CNN Model
class DeepCNN(nn.Module):
    def __init__(self):
        super(DeepCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(12, 64, kernel_size=3, padding=1),  # First conv layer
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # Second conv layer
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),  # Third conv layer
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),  # Fourth conv layer
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(512, 512, kernel_size=3, padding=1),  # Fifth conv layer
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 1 * 3, 1024),  # Adjust based on the final feature map size
            nn.ReLU(),
            nn.Dropout(dropout_prob_choice),
            nn.Linear(1024, 1)  # Output layer for regression (adjust for classification if needed)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create dataset and data loader
train_dataset = ARCDataset(num_samples=num_samples_choice)
train_loader = DataLoader(train_dataset, batch_size=batch_size_choice, shuffle=True)

# Create model and move to device
model = DeepCNN().to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate_choice)

num_epochs = num_epochs_choice
# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

Generating samples:   0%|          | 173/1000000000 [00:04<4910:43:23, 56.57it/s]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000172F4B6F020>>
Traceback (most recent call last):
  File "c:\Users\lerch\miniconda3\envs\arc\Lib\site-packages\ipykernel\ipkernel.py", line 794, in _clean_thread_parent_frames
    for identity in list(thread_to_parent_header.keys()):
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 
Generating samples:   0%|          | 459/1000000000 [00:23<14293:38:56, 19.43it/s]


KeyboardInterrupt: 